# Multinomial Naive Bayes Classification

In [11]:
import numpy as np
from keras.datasets import mnist
# load and split dataset into training and test sets
# cocatenate dataset into data features and labels
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)
data = np.zeros([70000, 784])
data[:60000, :] = x_train
data[60000:, :] = x_test
label = np.zeros(70000)
label[:60000] = y_train
label[60000:] = y_test

alpha = 100 #laplace smoothing value
n_classes = len(np.unique(label)) # number of classes in data
n_features = data.shape[1] # number of features in data

# compute class priors
def class_priors(label, n_classes):
	class_priors = []
	for i in range(n_classes):
		class_priors.append(np.sum(label == i) / len(label))
	return class_priors

# summarize stats by class
def summarize_by_class(data, label, n_classes):
	class_means = []
	class_stds = []
	for i in range(n_classes):
		class_data = data[label == i]
		class_means.append(np.mean(class_data, axis=0))
		class_stds.append(np.std(class_data, axis=0))
	return class_means, class_stds

# gaussian probability density function
def calculate_class_prob(x, mean, std):
	std = std + alpha
	class_prob = np.exp(-((x-mean)**2)/(2*std)) / np.sqrt(2*np.pi*(std))
	return class_prob

# predict function
def predict(x, summaries, class_priors, n_classes):
    class_probs = []
    class_means, class_stds = summaries
    for i in range(n_classes):
        prior = class_priors[i]
        mean = class_means[i]
        std = class_stds[i]
        class_prob = np.sum(np.log(calculate_class_prob(x, mean, std)))
        class_probs.append(class_prob + np.log(prior))
    return np.argmax(class_probs)

# naive Bayes
def naive_bayes(test, data, label, n_classes):
	predictions = []
	summaries = summarize_by_class(data, label, n_classes)
	priors = class_priors(label, n_classes)
	for i in range(len(test)):
		predictions.append(predict(test[i], summaries, priors, n_classes))
	return predictions

# evaluate accuracy
accuracy = np.sum(naive_bayes(x_test, data, label, n_classes) == y_test) / len(y_test) * 100
print(f'Accuracy: {accuracy:.2f}%')


Accuracy: 84.40%
